# SEO Keyword-to-Content Match Analyzer — V4
**For each keyword, determine whether existing content should target it, needs on-page optimization, or requires new content.**

### The model
Keywords are first classified by their own intrinsic search intent, then grouped into thematic clusters *within* that intent (so keywords with different intents can never merge, even if their embeddings are close — e.g. "CPQ" and "CPQ pricing"). Each cluster's representative keyword is then scored against your site's best-matching URL on three axes:

| Axis | Answers | Source |
|---|---|---|
| **Semantic match** | Will Google/AI platforms see this page as relevant? | Embedding cosine similarity |
| **Literal match** | Will searchers see this as a match (title/H1/H2/URL/first words)? | Position-weighted string match |
| **Intent match** | Does this page's type & specificity fit what the searcher wants? | ChatGPT (OpenAI API) classification |

Cell 15 also flags **thin-content risk** — a page that's otherwise a strong relevance match (No Action / On-Page Optimization) but is thin relative to this site's own crawled corpus, or below an absolute word-count floor. This is a self-calibrating relative check per site, not a fixed number applied uniformly across every client.

Cell 13 also generates a **searcher-intent summary** per cluster — a one-sentence, plain-English description of what someone searching that keyword actually wants (e.g. "sneaker cleaning tips" -> "A list of actionable steps and suggestions for better care and cleaning of sneakers."). This is based on the keyword alone, not the matched page, so it stays useful as a content-brief starting point even when the recommendation is "New Content."

Those combine into a recommendation per cluster (then propagated to every keyword in it):
- **Intent mismatch → New Content** (this gate overrides everything else)
- **Specificity mismatch → New Content** (a separate gate — a page can be the right TYPE of content while still being the wrong SCOPE, e.g. a narrow service page matched to a broad category keyword)
- **High semantic + High literal → No Action** (review performance separately)
- **High semantic + Low literal → On-Page Optimization**
- **Low semantic (regardless of literal) → New Content** (+ cannibalization flag if literal was high)

### Why intent-aware clustering
Pure semantic clustering has no concept of commercial intent — it groups on embedding proximity alone, and short keyword phrases often cluster on their shared head term rather than the intent-signaling words around it. "CPQ" and "CPQ pricing" can sit close enough in embedding space to merge even though one is informational and the other is bottom-funnel commercial. Classifying each keyword's intrinsic intent *before* clustering, and only allowing clusters to form within an intent group, closes that gap structurally rather than relying on catching it downstream. This step is also nearly free (no page context needed, so batches can be large) — see Cell 9.

### Why cluster first (for the paid steps)
Scoring every keyword individually against URLs and through GPT means near-duplicate keywords can get slightly different recommendations from embedding noise, and each one separately proposing "build new content." Clustering groups those together before the expensive steps run, so each cluster is scored/classified once, and every keyword in it gets a single, consistent call.

**Important caveat:** clustering *within* an intent group still relies on embedding similarity to catch true near-duplicates, which can't always tell "close variant of the same search" apart from "topically related but meaningfully different specificity" (the sneaker-vs-shoe problem — this is now caught by the specificity gate in Cell 14 rather than clustering itself). Cell 10 prints your largest clusters for a sanity check — review them and adjust `CLUSTER_SIMILARITY_THRESHOLD` in Cell 6 before continuing, since this happens before the paid API calls.

### Workflow
1. Install dependencies
2. Set up your OpenAI API key (two-stage: file upload or masked text entry)
3. Load embedding model + OpenAI client
4. Upload URLs list (CSV, column: `URL`)
5. Upload Keywords list (CSV, column: `Keyword`; optional `Search Volume` / `Keyword Difficulty` columns are auto-detected and used for cluster naming and rollups)
6. **Set match-quality, clustering, and few-shot config** (placeholder best-practice defaults — adjustable)
7. Crawl URLs, generate page embeddings, capture raw fields + word count
8. Generate keyword embeddings
9. **Classify each keyword's intrinsic intent** (informational/commercial/transactional/navigational) — cheap, page-independent, runs on every keyword
10. **Cluster keywords into thematic groups, within intent** — the representative (and cluster name) is the highest-search-volume keyword when volume data is available, falling back to centroid-closest otherwise. Review the printed sample clusters here
11. Semantic match (cluster representative → best-matching URL)
12. Literal match scoring
13. Intent+specificity classification of the matched page (OpenAI API — billed once per cluster, not per keyword)
14. Combine into final recommendation (per cluster)
15. Expand cluster recommendations back out to every keyword + build a cluster-level summary
16. Export to Excel (per-keyword sheet + cluster-summary sheet)
17. *(Optional)* Spot-check a single keyword and its cluster peers

### Before you run this
- **OpenAI API key required** for Cells 9 and 13. Upload a plain text file containing just your key (no quotes), or type it into a hidden prompt when asked.
- **Cost note:** both API-calling steps are cheap in absolute terms — typically well under a few dollars even at tens of thousands of keywords — since `gpt-4o-mini` is inexpensive and both steps batch aggressively. You'll see batch counts printed before each runs.
- **Input files:**
  - URLs file: CSV or Excel with a `URL` column
  - Keywords file: CSV or Excel with a `Keyword` column

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Run once per Colab session
# ============================================================
%%capture
%pip install aiohttp sentence-transformers beautifulsoup4 tqdm xlsxwriter openpyxl openai scikit-learn


In [ ]:
# ============================================================
# CELL 2 — OPENAI API KEY SETUP (two-stage)
# Stage 1: look for a local key file, or offer to upload one
#          (the file should contain ONLY the raw key, no quotes)
# Stage 2: fall back to a hidden manual text prompt
# ============================================================
import os
from getpass import getpass

API_KEY_FILENAME = "openai_api_key.txt"

def load_or_prompt_api_key():
    # Stage 1a — already sitting on disk from a prior run this session
    if os.path.exists(API_KEY_FILENAME):
        with open(API_KEY_FILENAME, "r") as f:
            key = f.read().strip()
        if key:
            print(f"Loaded API key from '{API_KEY_FILENAME}'.")
            return key

    # Stage 1b — offer to upload a key file
    print("No local API key file found.")
    choice = input("Upload a text file containing your OpenAI API key? (y/n): ").strip().lower()
    if choice == "y":
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            with open(fname, "r") as f:
                key = f.read().strip()
            if key:
                if fname != API_KEY_FILENAME:
                    os.rename(fname, API_KEY_FILENAME)
                print(f"API key loaded and saved as '{API_KEY_FILENAME}' for this session.")
                return key
            print("Uploaded file was empty — falling back to manual entry.")

    # Stage 2 — hidden manual entry
    print("Enter your OpenAI API key below (input is hidden).")
    key = getpass("OpenAI API Key: ").strip()
    if not key:
        raise ValueError("No API key provided. Cell 12 (intent classification) will fail without one.")
    return key

OPENAI_API_KEY = load_or_prompt_api_key()
print("API key set for this session.")


In [ ]:
# ============================================================
# CELL 3 — IMPORTS, EMBEDDING MODEL, OPENAI CLIENT
# ============================================================
import os
import re
import json
import time
import math
import string
import pickle
import asyncio
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import aiohttp
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from tqdm.asyncio import tqdm_asyncio
from tqdm.notebook import tqdm
from google.colab import files
from openai import OpenAI

# --- Embedding model ---
# all-MiniLM-L6-v2: fast, lightweight, 256-token context window.
# Upgrade to 'all-mpnet-base-v2' for higher accuracy / longer context
# (384 tokens) at ~3x slower encoding.
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print(f"Embedding model ready. Max sequence length: {model.max_seq_length} tokens.")
print("Note: text beyond this token limit is truncated before embedding — heavy")
print("title/H1/H2 weighting (see Cell 7) can eat into the budget available for body text.")

# --- OpenAI client (intent classification, Cell 12) ---
client = OpenAI(api_key=OPENAI_API_KEY)
GPT_MODEL = "gpt-4o-mini"   # cost-effective, sufficient for intent classification
print(f"OpenAI client ready. Using model: {GPT_MODEL}")


In [ ]:
# ============================================================
# CELL 4 — UPLOAD URLS FILE
# Required column: URL
# Accepts: CSV or Excel (.xlsx)
# ============================================================
print("Upload your URLs file (requires a 'URL' column)...")
uploaded_urls = files.upload()
url_file_name = list(uploaded_urls.keys())[0]

if url_file_name.endswith(".xlsx"):
    url_frame = pd.read_excel(url_file_name)
else:
    url_frame = pd.read_csv(url_file_name, encoding="utf-8-sig")

url_frame.columns = ["URL" if c.lower() == "url" else c for c in url_frame.columns]

if "URL" not in url_frame.columns:
    raise ValueError(f"Missing required column: 'URL'. Found: {list(url_frame.columns)}")

url_frame = url_frame.dropna(subset=["URL"]).drop_duplicates(subset=["URL"]).reset_index(drop=True)
unique_urls = url_frame["URL"].tolist()

print(f"\nURLs file loaded: '{url_file_name}'")
print(f"  {len(unique_urls):,} unique URLs to crawl")


In [ ]:
# ============================================================
# CELL 5 — UPLOAD KEYWORDS FILE
# Required column: Keyword
# Optional columns (auto-detected, case-insensitive): search volume,
# keyword difficulty. If present, these are preserved and used in Cell 10
# (cluster representative selection) and Cell 15 (cluster-level
# aggregation: total volume, average difficulty).
# Accepts: CSV or Excel (.xlsx)
# ============================================================
print("Upload your Keywords file (requires a 'Keyword' column)...")
uploaded_kws = files.upload()
kw_file_name = list(uploaded_kws.keys())[0]

if kw_file_name.endswith(".xlsx"):
    kw_frame = pd.read_excel(kw_file_name)
else:
    kw_frame = pd.read_csv(kw_file_name, encoding="utf-8-sig")

kw_frame.columns = [
    "Keyword" if c.lower() in ["keyword", "keywords", "kw", "query"] else c
    for c in kw_frame.columns
]

if "Keyword" not in kw_frame.columns:
    raise ValueError(f"Missing required column: 'Keyword'. Found: {list(kw_frame.columns)}")

_VOLUME_ALIASES = {
    "volume", "search volume", "search_volume", "avg monthly searches",
    "avg. monthly searches", "monthly searches", "msv", "sv",
}
_DIFFICULTY_ALIASES = {
    "difficulty", "keyword difficulty", "kd", "seo difficulty", "difficulty score",
}

kw_frame.columns = [
    "Search_Volume" if c.lower().strip() in _VOLUME_ALIASES else c
    for c in kw_frame.columns
]
kw_frame.columns = [
    "Keyword_Difficulty" if c.lower().strip() in _DIFFICULTY_ALIASES else c
    for c in kw_frame.columns
]

HAS_VOLUME_DATA = "Search_Volume" in kw_frame.columns
HAS_DIFFICULTY_DATA = "Keyword_Difficulty" in kw_frame.columns

def _parse_numeric_field(series):
    """Handles common keyword-tool export formats: commas (1,900), percent
    signs (45%), and K/M suffixes (10K, 1.2M) — coerces anything else to NaN
    rather than failing."""
    def _clean(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().replace(",", "").replace("%", "")
        mult = 1
        if s.lower().endswith("k"):
            mult = 1_000
            s = s[:-1]
        elif s.lower().endswith("m"):
            mult = 1_000_000
            s = s[:-1]
        try:
            return float(s) * mult
        except ValueError:
            return np.nan
    return series.apply(_clean)

if HAS_VOLUME_DATA:
    kw_frame["Search_Volume"] = _parse_numeric_field(kw_frame["Search_Volume"])
if HAS_DIFFICULTY_DATA:
    kw_frame["Keyword_Difficulty"] = _parse_numeric_field(kw_frame["Keyword_Difficulty"])

kw_frame = kw_frame.dropna(subset=["Keyword"]).drop_duplicates(subset=["Keyword"]).reset_index(drop=True)
unique_keywords = kw_frame["Keyword"].tolist()

keyword_volume_dict = dict(zip(kw_frame["Keyword"], kw_frame["Search_Volume"])) if HAS_VOLUME_DATA else {}
keyword_difficulty_dict = dict(zip(kw_frame["Keyword"], kw_frame["Keyword_Difficulty"])) if HAS_DIFFICULTY_DATA else {}

PKL_KEYWORD_METADATA_FILE = "keywordMetadata.pkl"
with open(PKL_KEYWORD_METADATA_FILE, "wb") as f:
    pickle.dump({
        "keyword_volume_dict": keyword_volume_dict,
        "keyword_difficulty_dict": keyword_difficulty_dict,
        "HAS_VOLUME_DATA": HAS_VOLUME_DATA,
        "HAS_DIFFICULTY_DATA": HAS_DIFFICULTY_DATA,
    }, f)
files.download(PKL_KEYWORD_METADATA_FILE)

print(f"\nKeywords file loaded: '{kw_file_name}'")
print(f"  {len(unique_keywords):,} unique keywords")
print(f"  Sample: {unique_keywords[:5]}")
if HAS_VOLUME_DATA:
    print("  Search volume column detected — will be used to pick each cluster's")
    print("  representative keyword (highest volume) and to sum volume per cluster.")
else:
    print("  No search volume column detected — cluster representatives will fall back")
    print("  to centroid-based selection (Cell 10).")
if HAS_DIFFICULTY_DATA:
    print("  Keyword difficulty column detected — will be averaged per cluster.")


In [ ]:
# ============================================================
# CELL 6 — MATCH-QUALITY, CLUSTERING & INTENT-CLASSIFICATION CONFIG
# Placeholder values below are general best-practice starting points,
# NOT calibrated to your specific site/vertical. Before trusting this
# at scale, calibrate against GSC data: pull similarity/literal scores
# for keyword-URL pairs you KNOW already rank well, and see where those
# scores actually fall.
# ============================================================

# --- Semantic thresholds (cosine similarity, 0.0-1.0) ---
# For all-MiniLM-L6-v2, genuinely relevant KEYWORD-TO-PAGE pairs typically
# score ~0.45-0.75. Scores >= ~0.55 are usually strong topical matches;
# below ~0.35 usually indicates unrelated content.
SEMANTIC_STRONG_THRESHOLD = 0.55   # >= this => "High" semantic match
SEMANTIC_WEAK_THRESHOLD   = 0.35   # <  this => "Low" semantic match
# Scores between the two are "Moderate" — surfaced in the output, but folded
# into "Low" for the recommendation logic (see Cell 13) unless you tighten
# these two values toward each other.

# --- Literal thresholds (0-100 position-weighted score) ---
# Weighting: Title 40, H1 25, URL slug 15, first 100 words 10, H2s 10
# (see Cell 11 for exact logic)
LITERAL_STRONG_THRESHOLD = 50      # >= this => "High" literal match
LITERAL_WEAK_THRESHOLD   = 20      # <  this => "Low" literal match

# For the near-variant fallback match (Cell 12): what fraction of a
# keyword's CONTENT words (stopwords excluded — including question words
# like "what"/"how"/"why") must appear in a field for it to count as a
# literal near-match. Computed over content words only, so filler words
# can't pad the score — this keeps the requirement meaningfully strict
# regardless of keyword length, rather than getting looser for longer
# phrases (see Cell 12 for the concrete failure case this fixes).
# Raise toward 1.0 for near-exact content-word coverage; lower toward 0.5
# if long-tail keywords are scoring "Weak" literal too often on pages
# that are clearly good real-world matches.
LITERAL_TERM_MATCH_THRESHOLD = 0.75

# Below LITERAL_TERM_MATCH_THRESHOLD but still meaningfully overlapping, a
# field can earn PARTIAL credit instead of zero (Cell 12): a field with
# ~half its content words present shouldn't score identically to a field
# with none. LITERAL_PARTIAL_MATCH_THRESHOLD is the coverage floor to
# qualify as "partial"; LITERAL_PARTIAL_MATCH_WEIGHT is the fraction of
# that field's full weight a "partial" match earns.
LITERAL_PARTIAL_MATCH_THRESHOLD = 0.50   # >= this (but below full) => "partial"
LITERAL_PARTIAL_MATCH_WEIGHT = 0.40      # partial match earns this fraction of the field's weight

# --- Thin-content risk (Cell 15) ---
# Flags pages that are otherwise strong matches (Recommendation "No Action"
# or "On-Page Optimization" — i.e. semantic/literal/intent already look
# good) but may still be too thin to perform well. A flat word-count
# number is a poor yardstick across verticals (800 words is thin for a
# SaaS comparison guide, plenty for a local service page) — so the
# primary check is RELATIVE to this site's own crawled corpus
# (self-calibrating per client, no manual tuning needed per engagement).
# THIN_CONTENT_ABSOLUTE_FLOOR is a backstop for sites where the whole
# corpus runs thin, since a relative percentile alone wouldn't catch that.
THIN_CONTENT_PERCENTILE = 25          # flag pages below this percentile of the site's own word counts
THIN_CONTENT_ABSOLUTE_FLOOR = 300     # flag regardless of percentile if word count is below this

# A general SEO rule-of-thumb minimum for being competitive, independent of
# what this specific site's own pages look like — a simple, EDITABLE stand-
# in for real SERP-based benchmarking (which would need a genuine SERP API
# integration; see the note below). 900 is a reasonable starting default
# for informational/comprehensive content — adjust per client or vertical
# (lower for product/transactional pages that are typically shorter by
# design, higher for competitive commercial comparison content).
COMPETITIVE_WORD_COUNT_THRESHOLD = 900

# The percentile above is computed SEGMENTED by page type (Contextual_
# Keyword_Intent from Cell 13), not across the whole crawled site — a 400-
# word product page and a 3,000-word pillar guide shouldn't be judged
# against the same distribution. Only pages that were actually matched to
# at least one cluster have a type label to segment by. If a segment has
# fewer than MIN_THIN_CONTENT_SEGMENT_SIZE matched pages, its percentile
# would be too noisy to trust, so that page falls back to a site-wide
# baseline computed across all matched pages instead (still excludes
# never-matched pages like /privacy or /careers, which would otherwise
# skew the baseline short and make genuinely thin content look normal).
MIN_THIN_CONTENT_SEGMENT_SIZE = 5

# NOTE: this is still a SELF-REFERENTIAL baseline — it tells you whether a
# page is thin relative to the rest of THIS site, not whether it's thin
# relative to what's actually ranking for the keyword in the SERP. A
# competitive baseline would require pulling word counts from top-ranking
# competitor pages (a SERP API integration) — a bigger addition, not
# something this heuristic approximates.

# --- Clustering threshold (cosine similarity, 0.0-1.0) ---
# How similar two KEYWORDS must be to each other to land in the same
# thematic cluster. This is intentionally set HIGHER (tighter) than the
# semantic MATCH threshold above — keyword-to-keyword similarity tends to
# run higher than keyword-to-page similarity in general (both are short,
# low-noise text), and clustering is a stronger claim than matching: you're
# saying these are near-duplicates of the SAME search, not just related
# topics. Too loose and you'll merge genuinely different searcher intents
# (e.g. "sneaker cleaning tips" and "how to clean shoes"); too tight and
# you won't catch real variants/rephrasings. Review Cell 9's printed
# sample clusters and adjust before proceeding to the paid Cell 12 step.
CLUSTER_SIMILARITY_THRESHOLD = 0.78

# --- Intent classification batching ---
# Handled by GPT rather than a numeric threshold — see Cell 12.
# INTENT_BATCH_SIZE controls how many cluster representatives are sent
# per API call (NOT per keyword, now that clustering runs first).
INTENT_BATCH_SIZE = 12

# ============================================================
# FEW-SHOT EXAMPLES FOR INTENT CLASSIFICATION
# These are sent as worked user/assistant example turns before every real
# batch in Cell 12, to anchor GPT's judgment on macro-intent (informational/
# commercial/transactional/navigational) and specificity (match/broader/
# narrower/different) alignment. Every example adds a small fixed
# prompt-token cost to EVERY batch call, so keep the total list reasonably
# small (roughly 4-10 total is a sane range) rather than exhaustive.
#
# FEWSHOT_EXAMPLES_DEFAULT are general-purpose and apply across verticals —
# edit or delete any that don't fit your use case.
#
# FEWSHOT_EXAMPLES_CUSTOM starts empty. Add your own vertical-specific
# examples here (e.g. orthodontics, CPQ/SaaS, home services) to sharpen
# classification for edge cases you've seen misclassified in your own
# output. Copy the template below, fill it in, and add it to the list.
#
# Allowed values:
#   keyword_intent:         informational | commercial | transactional | navigational
#   intent_alignment:       match | partial | mismatch
#   specificity_alignment:  match | broader | narrower | different
# ============================================================

FEWSHOT_EXAMPLES_DEFAULT = [
    {
        "keyword": "how does compound interest work",
        "page_url": "https://example.com/blog/compound-interest-explained",
        "page_title": "Compound Interest Explained: A Beginner's Guide",
        "page_h1": "How Compound Interest Works",
        "page_meta_description": "Learn what compound interest is, how it's calculated, and why it matters for long-term savings.",
        "page_snippet": "Compound interest is interest calculated on both the initial principal and the accumulated interest from previous periods.",
        "keyword_intent": "informational",
        "intent_alignment": "match",
        "specificity_alignment": "match",
        "reasoning": "Both keyword and page are informational explainers at the same level of depth.",
        "searcher_intent_summary": "A clear explanation of how compound interest works and why it matters for savings.",
    },
    {
        "keyword": "best budget laptops for college students",
        "page_url": "https://example.com/products/acme-ultrabook-14",
        "page_title": "Acme UltraBook 14 — Product Page",
        "page_h1": "Acme UltraBook 14",
        "page_meta_description": "Shop the Acme UltraBook 14. Lightweight, fast, and built for everyday use.",
        "page_snippet": "The Acme UltraBook 14 features a 14-inch display, 16GB RAM, and all-day battery life. Buy now and save 10%.",
        "keyword_intent": "commercial",
        "intent_alignment": "mismatch",
        "specificity_alignment": "narrower",
        "reasoning": "The keyword calls for a comparison across multiple laptops; a single product page can't fulfill that even though the topic overlaps.",
        "searcher_intent_summary": "A comparison of affordable laptop options suited to a college student's needs and budget.",
    },
    {
        "keyword": "buy wireless headphones online",
        "page_url": "https://example.com/blog/wireless-headphones-buying-guide",
        "page_title": "How to Choose the Right Wireless Headphones",
        "page_h1": "A Buyer's Guide to Wireless Headphones",
        "page_meta_description": "Tips for choosing wireless headphones based on sound quality, battery life, and fit.",
        "page_snippet": "Before buying wireless headphones, consider these factors: sound quality, battery life, comfort.",
        "keyword_intent": "transactional",
        "intent_alignment": "mismatch",
        "specificity_alignment": "match",
        "reasoning": "The keyword signals readiness to purchase; an educational buying-guide page doesn't support that action even though the topic matches.",
        "searcher_intent_summary": "A place to browse and purchase wireless headphones online.",
    },
    {
        "keyword": "email marketing",
        "page_url": "https://example.com/blog/welcome-email-templates",
        "page_title": "Welcome Email Templates for New Subscribers",
        "page_h1": "5 Welcome Email Templates That Convert",
        "page_meta_description": "Ready-to-use welcome email templates to onboard new subscribers.",
        "page_snippet": "A strong welcome email sets the tone for your relationship with new subscribers. Here are five templates.",
        "keyword_intent": "informational",
        "intent_alignment": "partial",
        "specificity_alignment": "narrower",
        "reasoning": "The keyword is a broad category term; this page covers one narrow tactic within it, not the full topic a searcher would expect.",
        "searcher_intent_summary": "A broad overview of email marketing as a discipline, including strategy, tools, and best practices.",
    },
]

# --- Add your own examples below. Template (copy, fill in, uncomment): ---
FEWSHOT_EXAMPLES_CUSTOM = [
    # {
    #     "keyword": "",
    #     "page_url": "",
    #     "page_title": "",
    #     "page_h1": "",
    #     "page_meta_description": "",
    #     "page_snippet": "",
    #     "keyword_intent": "informational",       # informational | commercial | transactional | navigational
    #     "intent_alignment": "match",              # match | partial | mismatch
    #     "specificity_alignment": "match",          # match | broader | narrower | different
    #     "reasoning": "",
    #     "searcher_intent_summary": "",             # one sentence, based on the KEYWORD only, not the page
    # },
]

_VALID_INTENTS = {"informational", "commercial", "transactional", "navigational"}
_VALID_INTENT_ALIGNMENT = {"match", "partial", "mismatch"}
_VALID_SPECIFICITY = {"match", "broader", "narrower", "different"}
_REQUIRED_FEWSHOT_KEYS = {"keyword", "keyword_intent", "intent_alignment", "specificity_alignment"}

def _validate_fewshot_example(example, source_label):
    missing = _REQUIRED_FEWSHOT_KEYS - example.keys()
    if missing:
        print(f"  Skipping {source_label} few-shot example — missing required field(s): {missing}")
        return False
    if example["keyword_intent"] not in _VALID_INTENTS:
        print(f"  Skipping {source_label} example ('{example['keyword']}') — invalid keyword_intent: '{example['keyword_intent']}'")
        return False
    if example["intent_alignment"] not in _VALID_INTENT_ALIGNMENT:
        print(f"  Skipping {source_label} example ('{example['keyword']}') — invalid intent_alignment: '{example['intent_alignment']}'")
        return False
    if example["specificity_alignment"] not in _VALID_SPECIFICITY:
        print(f"  Skipping {source_label} example ('{example['keyword']}') — invalid specificity_alignment: '{example['specificity_alignment']}'")
        return False
    return True

_valid_default = [ex for ex in FEWSHOT_EXAMPLES_DEFAULT if _validate_fewshot_example(ex, "default")]
_valid_custom = [ex for ex in FEWSHOT_EXAMPLES_CUSTOM if _validate_fewshot_example(ex, "custom")]
FEWSHOT_EXAMPLES = _valid_default + _valid_custom

print("Thresholds set:")
print(f"  Semantic   — High >= {SEMANTIC_STRONG_THRESHOLD}, Low < {SEMANTIC_WEAK_THRESHOLD}")
print(f"  Literal    — High >= {LITERAL_STRONG_THRESHOLD}, Low < {LITERAL_WEAK_THRESHOLD}")
print(f"  Clustering — similarity >= {CLUSTER_SIMILARITY_THRESHOLD}")
print(f"  Intent batch size: {INTENT_BATCH_SIZE} clusters/call")
# --- Standalone keyword-intent classification batching (Cell 9) ---
# Runs BEFORE clustering, on every individual keyword (not per cluster) —
# cheap because no page context is needed, so batches can be larger than
# the page-context step in Cell 13.
KEYWORD_INTENT_BATCH_SIZE = 40

# ============================================================
# FEW-SHOT EXAMPLES FOR STANDALONE KEYWORD-INTENT CLASSIFICATION (Cell 9)
# Simpler schema than the page-context examples above — just a keyword and
# its intrinsic intent, no page fields, since this step classifies the
# keyword alone. Used to partition keywords by intent BEFORE clustering, so
# "CPQ" (informational) and "CPQ pricing" (commercial) can never merge into
# the same cluster even if their embeddings are close.
# ============================================================

KEYWORD_INTENT_FEWSHOT_DEFAULT = [
    {"keyword": "how does compound interest work", "keyword_intent": "informational"},
    {"keyword": "best budget laptops for college students", "keyword_intent": "commercial"},
    {"keyword": "buy wireless headphones online", "keyword_intent": "transactional"},
    {"keyword": "nike.com", "keyword_intent": "navigational"},
    {"keyword": "email marketing", "keyword_intent": "informational"},
    {"keyword": "CPQ", "keyword_intent": "informational"},
    {"keyword": "CPQ pricing", "keyword_intent": "commercial"},
    {"keyword": "CPQ implementation services", "keyword_intent": "transactional"},
]

# --- Add your own examples below. Template (copy, fill in, uncomment): ---
KEYWORD_INTENT_FEWSHOT_CUSTOM = [
    # {"keyword": "", "keyword_intent": "informational"},  # informational | commercial | transactional | navigational
]

_REQUIRED_KEYWORD_INTENT_KEYS = {"keyword", "keyword_intent"}

def _validate_keyword_intent_example(example, source_label):
    missing = _REQUIRED_KEYWORD_INTENT_KEYS - example.keys()
    if missing:
        print(f"  Skipping {source_label} keyword-intent example — missing field(s): {missing}")
        return False
    if example["keyword_intent"] not in _VALID_INTENTS:
        print(f"  Skipping {source_label} example ('{example['keyword']}') — invalid keyword_intent: '{example['keyword_intent']}'")
        return False
    return True

_valid_kw_intent_default = [ex for ex in KEYWORD_INTENT_FEWSHOT_DEFAULT if _validate_keyword_intent_example(ex, "default keyword-intent")]
_valid_kw_intent_custom = [ex for ex in KEYWORD_INTENT_FEWSHOT_CUSTOM if _validate_keyword_intent_example(ex, "custom keyword-intent")]
KEYWORD_INTENT_FEWSHOT = _valid_kw_intent_default + _valid_kw_intent_custom

print(f"  Keyword-intent batch size: {KEYWORD_INTENT_BATCH_SIZE} keywords/call")
print(f"  Keyword-intent few-shot examples: {len(_valid_kw_intent_default)} default + {len(_valid_kw_intent_custom)} custom = {len(KEYWORD_INTENT_FEWSHOT)} total")
print(f"  Few-shot examples: {len(_valid_default)} default + {len(_valid_custom)} custom = {len(FEWSHOT_EXAMPLES)} total")


In [ ]:
# ============================================================
# CELL 7 — CRAWL PAGES, GENERATE PAGE EMBEDDINGS,
# CAPTURE RAW FIELDS FOR LITERAL MATCH + WORD COUNT
# Output: content_dict {url: {...}}
# ============================================================

MAX_CONCURRENT_REQUESTS = 20
REQUEST_TIMEOUT_SECONDS = 15
MAX_EMBED_CHARS = 5000        # cap on text fed to the embedding model
PKL_CONTENT_FILE = "contentDictionary.pkl"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; SEOAnalysisBot/1.0; +https://github.com/jasonmcgovern/gallery)"
}


def extract_page_fields(html: str, url: str) -> dict:
    """
    Parses raw HTML once and returns everything downstream cells need:
    weighted_text (semantic), raw fields (literal match), and word_count.
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    title = soup.title.get_text(strip=True) if soup.title else ""
    h1 = " ".join(t.get_text(strip=True) for t in soup.find_all("h1"))
    h2s = " ".join(t.get_text(strip=True) for t in soup.find_all("h2"))
    meta_tag = soup.find("meta", attrs={"name": "description"})
    meta_desc = meta_tag.get("content", "").strip() if meta_tag else ""

    body_text = soup.get_text(separator=" ", strip=True)
    body_text = re.sub(r"\s+", " ", body_text).strip()
    word_count = len(body_text.split())

    url_slug = urlparse(url).path.replace("-", " ").replace("_", " ").replace("/", " ").strip()

    weighted = (
        f"{title} {title} {title} "
        f"{h1} {h1} {h1} "
        f"{meta_desc} {meta_desc} "
        f"{h2s} {h2s} "
        f"{body_text[:MAX_EMBED_CHARS]}"
    ).strip()

    return {
        "title": title,
        "h1": h1,
        "h2s": h2s,
        "meta_desc": meta_desc,
        "url_slug": url_slug,
        "body_text": body_text[:MAX_EMBED_CHARS],
        "word_count": word_count,
        "weighted_text": weighted,
    }


async def fetch_url(session, url, semaphore):
    async with semaphore:
        try:
            async with session.get(
                url, timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
            ) as response:
                if response.status != 200:
                    return url, None, response.status
                # Use the response's own declared/detected charset instead of
                # forcing latin-1 (which mangles UTF-8 pages — curly quotes,
                # em dashes, accented characters, etc.)
                try:
                    html = await response.text()
                except UnicodeDecodeError:
                    raw = await response.read()
                    html = raw.decode("utf-8", errors="replace")
                return url, html, response.status
        except Exception as e:
            return url, None, f"error: {e}"


async def crawl_all(urls, max_concurrent=MAX_CONCURRENT_REQUESTS):
    semaphore = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession(headers=HEADERS) as session:
        tasks = [fetch_url(session, url, semaphore) for url in urls]
        results = await tqdm_asyncio.gather(*tasks, desc="Crawling pages")
    return results


print(f"Starting async crawl of {len(unique_urls):,} unique URLs...")
crawl_results = await crawl_all(unique_urls)

content_dict = {}
failed_urls = []

for url, html, status in crawl_results:
    if html is None:
        failed_urls.append((url, status))
        continue
    fields = extract_page_fields(html, url)
    if len(fields["body_text"]) < 100:
        failed_urls.append((url, "thin/empty content"))
        continue
    content_dict[url] = fields

print(f"\nCrawl complete: {len(content_dict):,} / {len(unique_urls):,} pages captured.")
if failed_urls:
    print(f"  {len(failed_urls):,} URLs failed or returned thin content:")
    for u, reason in failed_urls[:10]:
        print(f"    {u}  ({reason})")

if not content_dict:
    raise RuntimeError("No pages were captured. Check your URL list and network connection.")

print("\nGenerating page embeddings...")
crawled_urls = list(content_dict.keys())
texts = [content_dict[u]["weighted_text"] for u in crawled_urls]
page_embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

for i, u in enumerate(crawled_urls):
    content_dict[u]["embedding"] = page_embeddings[i]

with open(PKL_CONTENT_FILE, "wb") as f:
    pickle.dump(content_dict, f)

print(f"\nSaved {len(content_dict):,} page records (embeddings + raw fields + word count) to '{PKL_CONTENT_FILE}'.")
files.download(PKL_CONTENT_FILE)


In [ ]:
# ============================================================
# CELL 8 — GENERATE KEYWORD EMBEDDINGS
# Output: keyword_dict {keyword: {"embedding": vector}}
# ============================================================
PKL_KEYWORD_FILE = "keywordDictionary.pkl"

print(f"Generating embeddings for {len(unique_keywords):,} unique keywords...")
kw_embeddings = model.encode(
    unique_keywords, batch_size=256, show_progress_bar=True, normalize_embeddings=True
)

keyword_dict = {kw: {"embedding": kw_embeddings[i]} for i, kw in enumerate(unique_keywords)}

with open(PKL_KEYWORD_FILE, "wb") as f:
    pickle.dump(keyword_dict, f)

print(f"Saved {len(keyword_dict):,} keyword embeddings to '{PKL_KEYWORD_FILE}'.")
files.download(PKL_KEYWORD_FILE)


In [ ]:
# ============================================================
# CELL 9 — CLASSIFY EACH KEYWORD'S INTRINSIC INTENT
# Runs on every individual keyword (not per cluster), before clustering
# happens. Classifies informational/commercial/transactional/navigational
# intent from the keyword text ALONE — no page context, since no page has
# been matched yet at this point in the pipeline. This is what lets Cell 10
# partition keywords by intent before clustering, so keywords with
# different intents can never be merged into the same cluster even when
# their embeddings are close (e.g. "CPQ" vs. "CPQ pricing").
#
# Cheap relative to Cell 13's page-context classification: no page fields
# to send, so batches (KEYWORD_INTENT_BATCH_SIZE, Cell 6) can be larger.
# ============================================================

KEYWORD_INTENT_SYSTEM_PROMPT = """You are an SEO strategist classifying search intent for keywords.
For each keyword, classify its most likely searcher intent based on the keyword text alone —
no page context is provided; judge from typical searcher behavior for phrasing like this.

Categories:
- informational: the searcher wants to learn/understand something (how, what, why, guide, tips)
- commercial: the searcher is researching options before a purchase decision (best, top, reviews, vs, comparison)
- transactional: the searcher is ready to act now (buy, price, quote, near me, sign up, download)
- navigational: the searcher is looking for a specific brand, product, or destination by name

Respond ONLY with a JSON object: {"results": [...]}. Each item must have:
  "id": (echoed back exactly)
  "keyword_intent": one of "informational", "commercial", "transactional", "navigational"
No text outside the JSON object."""


def build_keyword_intent_messages(batch_keywords):
    messages = [{"role": "system", "content": KEYWORD_INTENT_SYSTEM_PROMPT}]
    for example in KEYWORD_INTENT_FEWSHOT:
        ex_in = {"keywords": [{"id": 0, "keyword": example["keyword"]}]}
        ex_out = {"results": [{"id": 0, "keyword_intent": example["keyword_intent"]}]}
        messages.append({"role": "user", "content": json.dumps(ex_in, ensure_ascii=False)})
        messages.append({"role": "assistant", "content": json.dumps(ex_out, ensure_ascii=False)})
    real_in = {"keywords": [{"id": i, "keyword": kw} for i, kw in enumerate(batch_keywords)]}
    messages.append({"role": "user", "content": json.dumps(real_in, ensure_ascii=False)})
    return messages


def classify_keyword_intent_batch(batch_keywords, max_retries=3):
    messages = build_keyword_intent_messages(batch_keywords)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                response_format={"type": "json_object"},
                messages=messages,
                temperature=0,
            )
            parsed = json.loads(response.choices[0].message.content)
            by_id = {r["id"]: r.get("keyword_intent") for r in parsed.get("results", [])}
            return [by_id.get(i, "unknown") for i in range(len(batch_keywords))]
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  Keyword-intent batch failed after {max_retries} attempts: {e}")
                return ["unknown"] * len(batch_keywords)
            time.sleep(2 ** attempt)


PKL_KEYWORD_INTENT_FILE = "keywordIntentDictionary.pkl"

n_batches = (len(unique_keywords) + KEYWORD_INTENT_BATCH_SIZE - 1) // KEYWORD_INTENT_BATCH_SIZE
print(f"Classifying intrinsic intent for {len(unique_keywords):,} keywords in {n_batches:,} "
      f"batches of up to {KEYWORD_INTENT_BATCH_SIZE} — this calls the OpenAI API and will incur "
      f"(small) cost.")

keyword_intent_dict = {}
for i in tqdm(range(0, len(unique_keywords), KEYWORD_INTENT_BATCH_SIZE), desc="Classifying keyword intent", total=n_batches):
    batch = unique_keywords[i:i + KEYWORD_INTENT_BATCH_SIZE]
    results = classify_keyword_intent_batch(batch)
    for kw, intent in zip(batch, results):
        keyword_intent_dict[kw] = intent

with open(PKL_KEYWORD_INTENT_FILE, "wb") as f:
    pickle.dump(keyword_intent_dict, f)
files.download(PKL_KEYWORD_INTENT_FILE)

print("\nIntrinsic keyword-intent classification complete.")
print(pd.Series(keyword_intent_dict).value_counts().to_string())


In [ ]:
# ============================================================
# CELL 10 — CLUSTER KEYWORDS INTO THEMATIC GROUPS, WITHIN INTENT
# Groups keywords that are near-duplicates/variants of the same underlying
# search — but ONLY within the same intrinsic intent (Cell 9), so keywords
# with different intents can never merge into the same cluster even if
# their embeddings are close (e.g. "CPQ" vs. "CPQ pricing").
#
# CAUTION: even within an intent group, pure embedding similarity can still
# merge keywords that are topically close but meaningfully different in
# SPECIFICITY (e.g. "sneaker cleaning tips" vs. the broader "how to clean
# shoes" — both plausibly "informational"). CLUSTER_SIMILARITY_THRESHOLD
# (Cell 6) is set tighter than the semantic MATCH threshold for exactly
# this reason, and the specificity gate in Cell 14 catches what slips
# through. Review the printed sample clusters below before continuing —
# this runs BEFORE the paid Cell 13 calls, so it's a free checkpoint.
# ============================================================
PKL_CLUSTER_FILE = "clusterAssignments.pkl"

if "keyword_intent_dict" not in globals() or not keyword_intent_dict:
    _ki = load_pkl_if_needed("keyword_intent_dict", PKL_KEYWORD_INTENT_FILE) if "load_pkl_if_needed" in globals() else None
    if _ki is None:
        if not os.path.exists(PKL_KEYWORD_INTENT_FILE):
            print(f"'{PKL_KEYWORD_INTENT_FILE}' not found. Please upload it now...")
            uploaded_pkl = files.upload()
            os.rename(list(uploaded_pkl.keys())[0], PKL_KEYWORD_INTENT_FILE)
        with open(PKL_KEYWORD_INTENT_FILE, "rb") as f:
            keyword_intent_dict = pickle.load(f)
    else:
        keyword_intent_dict = _ki

if "keyword_volume_dict" not in globals():
    PKL_KEYWORD_METADATA_FILE = "keywordMetadata.pkl"
    if not os.path.exists(PKL_KEYWORD_METADATA_FILE):
        print(f"'{PKL_KEYWORD_METADATA_FILE}' not found. Please upload it now...")
        uploaded_pkl = files.upload()
        os.rename(list(uploaded_pkl.keys())[0], PKL_KEYWORD_METADATA_FILE)
    with open(PKL_KEYWORD_METADATA_FILE, "rb") as f:
        _kw_meta = pickle.load(f)
    keyword_volume_dict = _kw_meta["keyword_volume_dict"]
    keyword_difficulty_dict = _kw_meta["keyword_difficulty_dict"]
    HAS_VOLUME_DATA = _kw_meta["HAS_VOLUME_DATA"]
    HAS_DIFFICULTY_DATA = _kw_meta["HAS_DIFFICULTY_DATA"]

kw_list_all = list(keyword_dict.keys())   # same order as unique_keywords
kw_embed_matrix = np.stack([keyword_dict[kw]["embedding"] for kw in kw_list_all])  # (K, D)
kw_intent_labels = [keyword_intent_dict.get(kw, "unknown") for kw in kw_list_all]

# Embeddings are normalized, so cosine distance = 1 - cosine similarity.
distance_threshold = 1 - CLUSTER_SIMILARITY_THRESHOLD

# Partition keyword indices by intrinsic intent, then cluster each
# partition independently — this is what makes cross-intent merges
# structurally impossible rather than just something we check for after.
intent_partitions = {}
for idx, intent in enumerate(kw_intent_labels):
    intent_partitions.setdefault(intent, []).append(idx)

keyword_to_cluster_id = {}
cluster_id_counter = 0

for intent_label, idxs in intent_partitions.items():
    if len(idxs) == 1:
        # AgglomerativeClustering needs >= 2 samples; a singleton just gets
        # its own cluster directly.
        keyword_to_cluster_id[kw_list_all[idxs[0]]] = cluster_id_counter
        cluster_id_counter += 1
        continue

    sub_matrix = kw_embed_matrix[idxs]
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric="cosine",
        linkage="average",
    )
    sub_labels = clusterer.fit_predict(sub_matrix)

    local_to_global = {}
    for local_pos, label in enumerate(sub_labels):
        if label not in local_to_global:
            local_to_global[label] = cluster_id_counter
            cluster_id_counter += 1
        global_idx = idxs[local_pos]
        keyword_to_cluster_id[kw_list_all[global_idx]] = local_to_global[label]

cluster_labels_final = [keyword_to_cluster_id[kw] for kw in kw_list_all]
cluster_df = pd.DataFrame({
    "Keyword": kw_list_all,
    "Cluster_ID": cluster_labels_final,
    "Keyword_Intent": kw_intent_labels,   # per-KEYWORD, not broadcast — genuine individual granularity
})
cluster_df["Search_Volume"] = cluster_df["Keyword"].map(keyword_volume_dict) if HAS_VOLUME_DATA else np.nan
cluster_df["Keyword_Difficulty"] = cluster_df["Keyword"].map(keyword_difficulty_dict) if HAS_DIFFICULTY_DATA else np.nan

cluster_sizes = cluster_df["Cluster_ID"].value_counts().to_dict()
cluster_df["Cluster_Size"] = cluster_df["Cluster_ID"].map(cluster_sizes)


def pick_cluster_representative(group_keywords, member_vecs):
    """
    Picks the cluster's representative/named keyword. Prefers the HIGHEST
    SEARCH VOLUME keyword in the group (ties broken by centroid proximity),
    since that's the keyword that should drive scoring and naming when
    real demand data is available. Falls back to pure centroid-closest
    ("medoid") selection if no search volume column was found in the
    Keywords file (Cell 5) — same behavior as before volume support existed.
    """
    if HAS_VOLUME_DATA:
        volumes = np.array([keyword_volume_dict.get(k, np.nan) for k in group_keywords], dtype=float)
        volumes = np.nan_to_num(volumes, nan=0.0)
        if volumes.max() > 0:
            candidates = np.where(volumes == volumes.max())[0]
            if len(candidates) == 1:
                return group_keywords[int(candidates[0])]
            centroid = member_vecs.mean(axis=0)
            norm = np.linalg.norm(centroid)
            if norm > 0:
                centroid = centroid / norm
            sims = member_vecs[candidates].dot(centroid)
            return group_keywords[int(candidates[int(np.argmax(sims))])]
        # No non-zero volume anywhere in this specific group (e.g. all NaN
        # for this cluster even though the file had a volume column) —
        # fall through to centroid selection below.

    centroid = member_vecs.mean(axis=0)
    norm = np.linalg.norm(centroid)
    if norm > 0:
        centroid = centroid / norm
    sims = member_vecs.dot(centroid)
    return group_keywords[int(np.argmax(sims))]


representatives = {}
for cid, group in cluster_df.groupby("Cluster_ID"):
    group_keywords = group["Keyword"].tolist()
    member_positions = [kw_list_all.index(k) for k in group_keywords]
    member_vecs = kw_embed_matrix[member_positions]
    representatives[cid] = pick_cluster_representative(group_keywords, member_vecs)

cluster_df["Cluster_Representative_Keyword"] = cluster_df["Cluster_ID"].map(representatives)

with open(PKL_CLUSTER_FILE, "wb") as f:
    pickle.dump({"cluster_df": cluster_df, "representatives": representatives}, f)
files.download(PKL_CLUSTER_FILE)

n_clusters = cluster_df["Cluster_ID"].nunique()
print(f"Clustered {len(kw_list_all):,} keywords into {n_clusters:,} thematic groups within "
      f"{len(intent_partitions):,} intent partitions (similarity threshold: {CLUSTER_SIMILARITY_THRESHOLD}).")
print(f"  Compression: {len(kw_list_all):,} keywords -> {n_clusters:,} groups to score "
      f"in Cells 11-14 (vs. scoring every keyword individually).")
print(f"  Representative selection: {'highest search volume (centroid tie-break)' if HAS_VOLUME_DATA else 'centroid-closest (no volume data found)'}.\n")

print("Largest clusters — sanity-check these before continuing. If any group looks")
print("like it's merging genuinely different SPECIFICITY (not intent — that's now")
print("structurally prevented), raise CLUSTER_SIMILARITY_THRESHOLD in Cell 6 and")
print("re-run this cell:\n")

top_clusters = cluster_df["Cluster_ID"].value_counts().head(8).index
for cid in top_clusters:
    cluster_rows = cluster_df[cluster_df["Cluster_ID"] == cid]
    members = cluster_rows["Keyword"].tolist()
    rep = representatives[cid]
    intent = cluster_rows["Keyword_Intent"].iloc[0]
    print(f"  Cluster {cid} ({len(members)} keywords, intent: {intent}) — representative: '{rep}'")
    for m in members[:8]:
        marker = "*" if m == rep else " "
        vol = keyword_volume_dict.get(m) if HAS_VOLUME_DATA else None
        vol_str = f" (vol: {int(vol):,})" if vol is not None and not pd.isna(vol) else ""
        print(f"    {marker} {m}{vol_str}")
    if len(members) > 8:
        print(f"      ... and {len(members) - 8} more")
    print()


In [ ]:
# ============================================================
# CELL 11 — SEMANTIC MATCH (cluster-centric)
# Scores each cluster's REPRESENTATIVE keyword against every URL, once per
# cluster rather than once per keyword. Every keyword in a cluster inherits
# its cluster's Best_URL/Semantic_Score in Cell 14.
# ============================================================

def load_pkl_if_needed(var_name, filename):
    if var_name in globals() and globals()[var_name] is not None:
        return globals()[var_name]
    if not os.path.exists(filename):
        print(f"'{filename}' not found. Please upload it now...")
        uploaded_pkl = files.upload()
        os.rename(list(uploaded_pkl.keys())[0], filename)
    with open(filename, "rb") as f:
        data = pickle.load(f)
    print(f"Loaded '{filename}'.")
    return data

content_dict = load_pkl_if_needed("content_dict", PKL_CONTENT_FILE)
keyword_dict = load_pkl_if_needed("keyword_dict", PKL_KEYWORD_FILE)

if "cluster_df" not in globals() or "representatives" not in globals():
    _cluster_state = load_pkl_if_needed("_cluster_state", PKL_CLUSTER_FILE)
    cluster_df = _cluster_state["cluster_df"]
    representatives = _cluster_state["representatives"]

url_list = list(content_dict.keys())
url_matrix = np.stack([content_dict[u]["embedding"] for u in url_list])  # (U, D)

representative_items = sorted(representatives.items())  # [(cluster_id, keyword), ...]

semantic_rows = []
for cid, kw in tqdm(representative_items, desc="Scoring cluster representatives against URLs"):
    kw_vec = np.array(keyword_dict[kw]["embedding"]).flatten()
    scores = url_matrix.dot(kw_vec)  # (U,)
    order = np.argsort(scores)[::-1]

    best_idx = order[0]
    runner_idx = order[1] if len(order) > 1 else None

    semantic_rows.append({
        "Cluster_ID": cid,
        "Cluster_Representative_Keyword": kw,
        "Best_URL": url_list[best_idx],
        "Semantic_Score": round(float(scores[best_idx]), 4),
        "Runner_Up_URL": url_list[runner_idx] if runner_idx is not None else None,
        "Runner_Up_Score": round(float(scores[runner_idx]), 4) if runner_idx is not None else None,
    })

semantic_df = pd.DataFrame(semantic_rows)

# Ambiguous match: runner-up is nearly as strong as the best match — two
# pages are competing for this cluster's territory. Worth a manual look.
semantic_df["Ambiguous_Match"] = (
    semantic_df["Runner_Up_Score"].notna() &
    ((semantic_df["Semantic_Score"] - semantic_df["Runner_Up_Score"]) < 0.03)
)

print(f"\nSemantic matching complete for {len(semantic_df):,} clusters "
      f"(representing {len(cluster_df):,} total keywords).")
print(semantic_df[["Cluster_Representative_Keyword", "Best_URL", "Semantic_Score"]].head(5).to_string(index=False))


In [ ]:
# ============================================================
# CELL 12 — LITERAL MATCH (position-weighted string match, cluster-centric)
# Checks whether each cluster's representative keyword phrase (or a close
# variant) appears in high-value fields of that cluster's matched page.
# Weights: Title 40, H1 25, URL slug 15, first 100 words 10, H2s 10
# (H2 match = the page has a dedicated subheading on this topic — a
# stronger structural signal than an incidental body mention. Raw body
# presence is still tracked below for diagnostic purposes, just not
# scored, since a single buried mention doesn't help a searcher scanning
# the page confirm relevance.)
#
# Each field gets one of three match levels rather than a binary
# yes/no, based on the fraction of CONTENT words (stopwords excluded,
# including question words like "what"/"how"/"why") present:
#   "full"    — exact phrase, or >= LITERAL_TERM_MATCH_THRESHOLD coverage
#               -> field earns its FULL weight
#   "partial" — >= LITERAL_PARTIAL_MATCH_THRESHOLD coverage (but below full)
#               -> field earns LITERAL_PARTIAL_MATCH_WEIGHT of its weight
#   "none"    — below the partial threshold -> field earns 0
# A field that's clearly related to the keyword but doesn't fully cover it
# shouldn't score identically to a field with zero overlap — this gives
# it graduated credit instead. Computing coverage over content words only
# (not raw words) is what keeps this meaningful regardless of keyword
# length — see the thresholds and comments in Cell 6.
# ============================================================

# A compact, self-contained stopword list (no extra dependency) — tuned
# for SEO-keyword filler words, including interrogatives common in
# informational long-tail queries.
_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "of", "at",
    "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up",
    "down", "in", "out", "on", "off", "over", "under", "again", "further",
    "is", "are", "was", "were", "be", "been", "being", "do", "does", "did",
    "have", "has", "had", "having", "this", "that", "these", "those", "i",
    "you", "he", "she", "it", "we", "they", "my", "your", "his", "her",
    "its", "our", "their", "as", "can", "will", "just", "should", "now",
    "what", "how", "why", "which", "who", "where", "when",
}

_MATCH_LEVEL_WEIGHT = {
    "full": 1.0,
    "partial": None,   # filled in below from LITERAL_PARTIAL_MATCH_WEIGHT (Cell 6)
    "none": 0.0,
}
_MATCH_LEVEL_WEIGHT["partial"] = LITERAL_PARTIAL_MATCH_WEIGHT

def normalize_text(t):
    t = t.lower()
    t = t.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", t).strip()

def classify_field_match(haystack, keyword):
    """
    Returns "full", "partial", or "none" for how strongly the keyword
    matches this field. An exact phrase match is always "full". Otherwise
    based on the fraction of CONTENT words (stopwords excluded) present —
    computed as integer hit-count thresholds (via ceil) rather than a raw
    float ratio comparison, to avoid floating-point edge cases at the
    boundary.
    """
    haystack_n = normalize_text(haystack)
    keyword_n = normalize_text(keyword)
    if not keyword_n:
        return "none"
    if keyword_n in haystack_n:
        return "full"

    kw_terms = keyword_n.split()
    if not kw_terms:
        return "none"

    content_terms = [t for t in kw_terms if t not in _STOPWORDS]
    if not content_terms:
        # Keyword was entirely stopwords (rare for real SEO keywords) —
        # fall back to the full term list rather than requiring zero terms.
        content_terms = kw_terms

    full_required = math.ceil(LITERAL_TERM_MATCH_THRESHOLD * len(content_terms))
    partial_required = math.ceil(LITERAL_PARTIAL_MATCH_THRESHOLD * len(content_terms))
    hits = sum(1 for term in content_terms if term in haystack_n)

    if hits >= full_required:
        return "full"
    elif hits >= partial_required:
        return "partial"
    else:
        return "none"

def literal_match_score(keyword, fields):
    body = fields.get("body_text", "")
    words = body.split()
    first_100 = " ".join(words[:100])
    rest_body = " ".join(words[100:])   # tracked for diagnostics, not scored

    field_weights = [
        ("title", fields.get("title", ""), 40),
        ("h1", fields.get("h1", ""), 25),
        ("url_slug", fields.get("url_slug", ""), 15),
        ("first_100_words", first_100, 10),
        ("h2", fields.get("h2s", ""), 10),
    ]

    score = 0.0
    detail = {}
    for key, text, weight in field_weights:
        level = classify_field_match(text, keyword)
        detail[key] = level
        score += weight * _MATCH_LEVEL_WEIGHT[level]

    # Diagnostic only — NOT added to score.
    detail["body"] = classify_field_match(rest_body, keyword)

    return round(score, 1), detail


literal_rows = []
for _, row in semantic_df.iterrows():
    kw, url = row["Cluster_Representative_Keyword"], row["Best_URL"]
    fields = content_dict.get(url, {})
    score, detail = literal_match_score(kw, fields)
    literal_rows.append({
        "Cluster_Representative_Keyword": kw,
        "Literal_Score": score,
        "Literal_In_Title": detail.get("title", "none"),
        "Literal_In_H1": detail.get("h1", "none"),
        "Literal_In_URL": detail.get("url_slug", "none"),
        "Literal_In_First100Words": detail.get("first_100_words", "none"),
        "Literal_In_H2": detail.get("h2", "none"),
        "Literal_In_Body": detail.get("body", "none"),   # display-only, unscored
    })

literal_df = pd.DataFrame(literal_rows)
cluster_match_df = semantic_df.merge(literal_df, on="Cluster_Representative_Keyword", how="left")

print("Literal match scoring complete.")
print(cluster_match_df[["Cluster_Representative_Keyword", "Semantic_Score", "Literal_Score"]].head(5).to_string(index=False))


In [ ]:
# ============================================================
# CELL 13 — INTENT + SPECIFICITY MATCH OF THE PAGE, AND A SEARCHER-INTENT
# SUMMARY (via OpenAI API, cluster-centric)
# Answers a different question than Cell 9's classification: not "what is
# this keyword's own intent" but "does the MATCHED PAGE fit that intent and
# specificity." Output field is named Contextual_Keyword_Intent to keep it
# distinct from Cell 10's intrinsic, page-independent Keyword_Intent.
# For each cluster's representative keyword and its best-matching URL,
# classifies whether the page's type AND specificity align with what the
# searcher wants — not just whether the topic is related. Runs once per
# CLUSTER, not once per keyword, so this is where clustering saves real
# API cost.
#
# Also generates Searcher_Intent_Summary — a one-sentence plain-English
# description of what someone searching this cluster's keyword actually
# wants (e.g. "sneaker cleaning tips" -> "A list of actionable steps and
# suggestions for better care and cleaning of sneakers."). This is
# deliberately based on the KEYWORD alone, not a description of the
# matched page — so it stays useful as a content-brief seed even (and
# especially) when the recommendation is "New Content" and the matched
# page isn't the right one. Piggybacks on this same API call rather than
# adding a separate step, since the marginal cost of a few extra output
# tokens per item is negligible next to the fixed per-call overhead.
#
# Few-shot examples (FEWSHOT_EXAMPLES, defined in Cell 6 — defaults plus
# any custom ones you added) are sent as actual user/assistant message
# turns before the real batch. Worked examples anchor the model's judgment
# more reliably than definitions alone, and demonstrate the exact expected
# JSON shape at the same time.
# ============================================================

INTENT_SYSTEM_PROMPT = """You are an SEO strategist evaluating search intent alignment.
For each keyword/page pair, judge whether the page is the RIGHT TYPE of content and
the RIGHT LEVEL OF SPECIFICITY for someone searching that keyword — not just whether
the topic is related.

Consider:
- Macro intent: informational, commercial, transactional, or navigational — does the
  page's format match what the query calls for? (e.g. a blog post is a poor fit for a
  clearly transactional/buy-now query, even if topically related)
- Specificity: is the page written at the same level of specificity as the query, or is
  it broader/narrower? (e.g. a generic "how to clean shoes" page is a partial fit at best
  for "sneaker cleaning tips" — different specificity, possibly different audience. A
  narrow bottom-funnel service page is also a poor fit for a broad category keyword that
  calls for a comprehensive overview, even if the topic is technically related.)

You will also receive the page's URL (page_url). URL path segments are often a strong,
concrete signal of page type and specificity — e.g. "/blog/" or "/guides/" suggests
informational content, "/services/" or "/implementation/" suggests a narrow commercial
service page, "/pricing/" suggests transactional. Weigh this alongside the other fields,
not instead of them — a URL pattern is a hint, not a guarantee.

Also write a searcher_intent_summary: one plain-English sentence describing what someone
searching this KEYWORD is actually looking for. Base this ONLY on the keyword's own
topic and implied intent — do NOT describe or summarize the matched page's content. This
summary should be equally valid whether or not the matched page actually serves the
keyword well, since it may be used as a content-brief starting point for a page that
doesn't exist yet.

Respond ONLY with a JSON object: {"results": [...]}. Each item in "results" must have:
  "id": (the id provided in the input, echoed back exactly)
  "keyword_intent": one of "informational", "commercial", "transactional", "navigational"
  "intent_alignment": one of "match", "partial", "mismatch"
  "specificity_alignment": one of "match", "broader", "narrower", "different"
  "reasoning": a one-sentence explanation of the intent_alignment/specificity_alignment judgment
  "searcher_intent_summary": one plain-English sentence describing what the searcher wants
No text outside the JSON object."""


def fewshot_example_to_message_pair(example):
    """Converts one flat FEWSHOT_EXAMPLES dict (Cell 6) into the same
    {"pairs": [...]} / {"results": [...]} JSON shape used for real batches,
    so the worked examples demonstrate the exact expected format."""
    input_payload = {"pairs": [{
        "id": 0,
        "keyword": example["keyword"],
        "page_url": example.get("page_url", ""),
        "page_title": example.get("page_title", ""),
        "page_h1": example.get("page_h1", ""),
        "page_meta_description": example.get("page_meta_description", ""),
        "page_snippet": example.get("page_snippet", ""),
    }]}
    output_payload = {"results": [{
        "id": 0,
        "keyword_intent": example["keyword_intent"],
        "intent_alignment": example["intent_alignment"],
        "specificity_alignment": example["specificity_alignment"],
        "reasoning": example.get("reasoning", ""),
        "searcher_intent_summary": example.get("searcher_intent_summary", ""),
    }]}
    return input_payload, output_payload


def build_intent_batch_payload(batch_rows):
    items = []
    for i, row in enumerate(batch_rows):
        fields = content_dict.get(row["Best_URL"], {})
        items.append({
            "id": i,
            "keyword": row["Cluster_Representative_Keyword"],
            "page_url": row["Best_URL"],
            "page_title": fields.get("title", ""),
            "page_h1": fields.get("h1", ""),
            "page_meta_description": fields.get("meta_desc", ""),
            "page_snippet": fields.get("body_text", "")[:400],
        })
    return items


def build_intent_messages(user_content):
    messages = [{"role": "system", "content": INTENT_SYSTEM_PROMPT}]
    for example in FEWSHOT_EXAMPLES:
        ex_in, ex_out = fewshot_example_to_message_pair(example)
        messages.append({"role": "user", "content": json.dumps(ex_in, ensure_ascii=False)})
        messages.append({"role": "assistant", "content": json.dumps(ex_out, ensure_ascii=False)})
    messages.append({"role": "user", "content": user_content})
    return messages


def classify_intent_batch(batch_rows, max_retries=3):
    items = build_intent_batch_payload(batch_rows)
    user_content = json.dumps({"pairs": items}, ensure_ascii=False)
    messages = build_intent_messages(user_content)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                response_format={"type": "json_object"},
                messages=messages,
                temperature=0,
            )
            parsed = json.loads(response.choices[0].message.content)
            by_id = {r["id"]: r for r in parsed.get("results", [])}
            output = []
            for i in range(len(batch_rows)):
                r = by_id.get(i, {})
                output.append({
                    "Contextual_Keyword_Intent": r.get("keyword_intent"),  # renamed from "Keyword_Intent" to avoid
                    # colliding with the intrinsic per-keyword Keyword_Intent column from Cell 10
                    "Intent_Alignment": r.get("intent_alignment", "mismatch"),
                    "Specificity_Alignment": r.get("specificity_alignment"),
                    "Intent_Reasoning": r.get("reasoning", ""),
                    "Searcher_Intent_Summary": r.get("searcher_intent_summary", ""),
                })
            return output
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  Intent batch failed after {max_retries} attempts: {e}")
                return [{
                    "Contextual_Keyword_Intent": None,
                    "Intent_Alignment": "unknown",
                    "Specificity_Alignment": None,
                    "Intent_Reasoning": f"API error: {e}",
                    "Searcher_Intent_Summary": "",
                } for _ in batch_rows]
            time.sleep(2 ** attempt)


rows_as_dicts = cluster_match_df.to_dict("records")
n_batches = (len(rows_as_dicts) + INTENT_BATCH_SIZE - 1) // INTENT_BATCH_SIZE
print(f"Classifying intent for {len(rows_as_dicts):,} clusters "
      f"(covering {len(cluster_df):,} total keywords) in {n_batches:,} batches of up to "
      f"{INTENT_BATCH_SIZE} — this calls the OpenAI API and will incur cost.")
print(f"  ({len(FEWSHOT_EXAMPLES)} few-shot examples from Cell 6 are included in every "
      f"batch call, adding a small fixed prompt-token cost per call.)")

intent_results = []
for i in tqdm(range(0, len(rows_as_dicts), INTENT_BATCH_SIZE), desc="Classifying intent", total=n_batches):
    batch = rows_as_dicts[i:i + INTENT_BATCH_SIZE]
    intent_results.extend(classify_intent_batch(batch))

intent_df = pd.DataFrame(intent_results)
cluster_match_df = pd.concat([cluster_match_df.reset_index(drop=True), intent_df.reset_index(drop=True)], axis=1)

print("\nIntent classification complete.")
print(cluster_match_df[["Cluster_Representative_Keyword", "Intent_Alignment", "Specificity_Alignment"]].head(5).to_string(index=False))


In [ ]:
# ============================================================
# CELL 14 — COMBINE THE THREE AXES INTO A RECOMMENDATION (per cluster)
#
# Decision tree (run once per cluster):
#   1. Intent mismatch (or "partial", if INTENT_PARTIAL_AS_MISMATCH is
#      True) -> New Content.
#   2. Specificity mismatch (Specificity_Alignment != "match", if
#      REQUIRE_SPECIFICITY_MATCH is True) -> New Content. This is a
#      SEPARATE gate from intent: a page can be the right TYPE of content
#      (e.g. commercial page for a commercial query) while still being the
#      wrong SCOPE for the keyword (e.g. a narrow bottom-funnel service
#      page matched to a broad category term like "CPQ") — GPT classifies
#      both, and both need to hold for the page to be a genuine fit.
#   3. Intrinsic-vs-contextual intent disagreement (if
#      REQUIRE_INTRINSIC_CONTEXTUAL_INTENT_AGREEMENT is True) -> New
#      Content. Cell 9's Keyword_Intent (classified from the keyword ALONE,
#      before any page is involved) and Cell 13's Contextual_Keyword_Intent
#      (classified WITH the matched page in view) come from two
#      independent passes. When they disagree, GPT's own Intent_Alignment
#      field for that same call can still say "match" — Intent_Alignment
#      and Contextual_Keyword_Intent are two outputs from the same
#      response, not independently verified against each other, so a
#      single call isn't fully self-consistent. Cross-checking against
#      the keyword-only classification is what actually catches it. This
#      is what closes the gap that let a keyword like "CPQ" (intrinsic:
#      informational) matched to a commercial-context page slip through
#      as "No Action" even though the two intent reads disagreed.
#   Any of the three gates alone routes to New Content; the reason string
#   says which one(s) fired.
#   4. Otherwise, a 2x2 on the semantic/literal thresholds from Cell 6:
#        High semantic + High literal -> No Action
#        High semantic + Low literal  -> On-Page Optimization
#        Low semantic  + High literal -> New Content (+ cannibalization flag)
#        Low semantic  + Low literal  -> New Content
#   "Moderate" scores (between Weak and Strong thresholds) are folded into
#   "Low" here. Tighten SEMANTIC_WEAK_THRESHOLD / LITERAL_WEAK_THRESHOLD
#   toward the Strong thresholds in Cell 6 if you'd rather route moderate
#   scores into No Action / On-Page Optimization instead.
# ============================================================

# Treat a "partial" intent alignment as a mismatch (-> New Content) rather
# than passing it through to the semantic/literal grid. Default True, since
# a specificity gap usually does need dedicated content, not just a retitle.
INTENT_PARTIAL_AS_MISMATCH = True

# Require Specificity_Alignment == "match" for a page to clear the gate.
# Default True (strict): "broader", "narrower", and "different" all route
# to New Content. If this produces too many false positives once you're
# looking at real output (e.g. a broad pillar page reasonably serving a
# narrower long-tail keyword), set to False to fall back to intent-only
# gating, or narrow SPECIFICITY_MISMATCH_VALUES below to just the values
# you actually want to treat as disqualifying.
REQUIRE_SPECIFICITY_MATCH = True
SPECIFICITY_MISMATCH_VALUES = {"broader", "narrower", "different"}

# Require the cluster representative's intrinsic Keyword_Intent (Cell 9,
# keyword-only) to match Contextual_Keyword_Intent (Cell 13, keyword+page)
# for a page to clear the gate. Default True (strict). If this fires too
# often on cases where the disagreement is genuinely benign (e.g.
# informational vs. navigational for a branded term), set to False to
# rely on Intent_Alignment/Specificity_Alignment alone.
REQUIRE_INTRINSIC_CONTEXTUAL_INTENT_AGREEMENT = True


def classify_semantic(score):
    if score >= SEMANTIC_STRONG_THRESHOLD:
        return "High"
    elif score < SEMANTIC_WEAK_THRESHOLD:
        return "Low"
    return "Moderate"

def classify_literal(score):
    if score >= LITERAL_STRONG_THRESHOLD:
        return "High"
    elif score < LITERAL_WEAK_THRESHOLD:
        return "Low"
    return "Moderate"

def recommend(row):
    intent = row["Intent_Alignment"]
    specificity = row["Specificity_Alignment"]
    intrinsic_intent = row.get("Keyword_Intent")
    contextual_intent = row.get("Contextual_Keyword_Intent")

    intent_mismatch = (intent == "mismatch") or (intent == "partial" and INTENT_PARTIAL_AS_MISMATCH)
    specificity_mismatch = REQUIRE_SPECIFICITY_MATCH and specificity in SPECIFICITY_MISMATCH_VALUES
    intent_label_mismatch = (
        REQUIRE_INTRINSIC_CONTEXTUAL_INTENT_AGREEMENT
        and pd.notna(intrinsic_intent) and pd.notna(contextual_intent)
        and intrinsic_intent != contextual_intent
    )

    if intent_mismatch or specificity_mismatch or intent_label_mismatch:
        reasons = []
        if intent_mismatch:
            reasons.append(f"intent {intent}")
        if specificity_mismatch:
            reasons.append(f"specificity {specificity}")
        if intent_label_mismatch:
            reasons.append(f"keyword intent '{intrinsic_intent}' vs. page-context intent '{contextual_intent}'")
        return "New Content", "Mismatch — " + ", ".join(reasons)

    sem_high = classify_semantic(row["Semantic_Score"]) == "High"
    lit_high = classify_literal(row["Literal_Score"]) == "High"

    if sem_high and lit_high:
        return "No Action", "Strong semantic + literal match; review performance separately"
    elif sem_high and not lit_high:
        return "On-Page Optimization", "Topically strong but missing literal keyword signal in key fields"
    elif not sem_high and lit_high:
        return "New Content", "Literal match only — likely thin/tangential coverage; check for cannibalization"
    else:
        return "New Content", "Low semantic and literal match — genuine content gap"


# Bring in the representative keyword's own INTRINSIC Keyword_Intent
# (Cell 9/10, per-keyword, not previously available on cluster_match_df)
# so recommend() can actually compare it against Contextual_Keyword_Intent,
# rather than that comparison only happening as a downstream diagnostic
# after recommendations are already finalized.
_representative_intent_lookup = (
    cluster_df[cluster_df["Keyword"] == cluster_df["Cluster_Representative_Keyword"]]
    [["Cluster_Representative_Keyword", "Keyword_Intent"]]
    .drop_duplicates(subset=["Cluster_Representative_Keyword"])
)
cluster_match_df = cluster_match_df.merge(
    _representative_intent_lookup, on="Cluster_Representative_Keyword", how="left"
)

recommendations = cluster_match_df.apply(recommend, axis=1, result_type="expand")
cluster_match_df["Recommendation"] = recommendations[0]
cluster_match_df["Recommendation_Reason"] = recommendations[1]
cluster_match_df["Semantic_Bucket"] = cluster_match_df["Semantic_Score"].apply(classify_semantic)
cluster_match_df["Literal_Bucket"] = cluster_match_df["Literal_Score"].apply(classify_literal)

# Cannibalization risk, now at the CLUSTER level: different clusters
# (genuinely distinct topics, not just keyword variants of each other)
# landing on the same Best_URL with a "New Content" call — a stronger
# signal than the old per-keyword version, since these clusters aren't
# variants of each other by construction.
url_counts = cluster_match_df["Best_URL"].value_counts()
cluster_match_df["Shared_Best_URL_Cluster_Count"] = cluster_match_df["Best_URL"].map(url_counts)
cluster_match_df["Cannibalization_Risk"] = (
    (cluster_match_df["Recommendation"] == "New Content") &
    (cluster_match_df["Shared_Best_URL_Cluster_Count"] > 1)
)

print("Recommendation counts (per cluster):")
print(cluster_match_df["Recommendation"].value_counts().to_string())


In [ ]:
# ============================================================
# CELL 15 — EXPAND CLUSTER RESULTS BACK TO EVERY KEYWORD
# Joins the cluster-level recommendation onto every individual keyword
# (so the main export still shows one row per keyword), and builds a
# separate cluster-level summary for a content-planning view.
# ============================================================

# cluster_match_df also carries a Cluster_ID column (from Cell 10) — drop it
# before merging so pandas doesn't auto-suffix the duplicate into
# Cluster_ID_x/Cluster_ID_y and silently break every downstream reference
# to match_df["Cluster_ID"].
# cluster_match_df now also carries Cluster_ID (from Cell 11) AND
# Keyword_Intent (added in Cell 14, needed there for the intrinsic-vs-
# contextual gate) — both duplicate columns already present in cluster_df.
# Drop both before merging so pandas doesn't auto-suffix them into
# _x/_y and silently break downstream references to match_df["Cluster_ID"]
# or match_df["Keyword_Intent"].
match_df = cluster_df.merge(
    cluster_match_df.drop(columns=["Cluster_ID", "Keyword_Intent"]),
    on="Cluster_Representative_Keyword", how="left"
)

# Word count, attached for operator context.
match_df["Matched_Page_Word_Count"] = match_df["Best_URL"].map(
    lambda u: content_dict.get(u, {}).get("word_count")
)

# Thin-content risk: flags cases where relevance signals already look
# strong (No Action / On-Page Optimization) but the matched page may still
# be too thin to perform well. See Cell 6 for the threshold rationale.
# "New Content" rows are excluded — thinness isn't the diagnosis there,
# the page isn't the right one yet regardless of its length.
#
# The baseline is SEGMENTED by page type (Contextual_Keyword_Intent) so a
# short product page isn't judged against a long-form guide's typical
# length. Only pages that were actually matched to a cluster have a type
# label — that's also a deliberately narrower (and more relevant)
# population than "every crawled URL," since it excludes pages like
# /privacy or /careers that would otherwise skew the baseline artificially
# short.
_page_segment_lookup = (
    match_df[["Best_URL", "Contextual_Keyword_Intent"]]
    .dropna(subset=["Best_URL"])
    .groupby("Best_URL")["Contextual_Keyword_Intent"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
)

_segment_word_counts = {}
for _url, _segment in _page_segment_lookup.items():
    _wc = content_dict.get(_url, {}).get("word_count")
    if _wc is not None:
        _segment_word_counts.setdefault(_segment, []).append(_wc)

_segment_thresholds = {}
for _segment, _counts in _segment_word_counts.items():
    if len(_counts) >= MIN_THIN_CONTENT_SEGMENT_SIZE:
        _segment_thresholds[_segment] = np.percentile(_counts, THIN_CONTENT_PERCENTILE)

# Site-wide fallback for segments too small to trust on their own — still
# computed across matched pages only, not the full crawl.
_all_matched_word_counts = [wc for counts in _segment_word_counts.values() for wc in counts]
_site_fallback_threshold = (
    np.percentile(_all_matched_word_counts, THIN_CONTENT_PERCENTILE) if _all_matched_word_counts else 0
)

print(f"Thin-content baselines by page type ({THIN_CONTENT_PERCENTILE}th percentile of matched pages):")
for _segment, _counts in sorted(_segment_word_counts.items(), key=lambda kv: str(kv[0])):
    if _segment in _segment_thresholds:
        print(f"  {_segment}: {_segment_thresholds[_segment]:.0f} words (n={len(_counts)})")
    else:
        print(f"  {_segment}: only {len(_counts)} matched pages "
              f"(< {MIN_THIN_CONTENT_SEGMENT_SIZE}) — using site-wide fallback "
              f"({_site_fallback_threshold:.0f} words) instead")

def _thin_content_threshold_for_url(url):
    segment = _page_segment_lookup.get(url)
    return _segment_thresholds.get(segment, _site_fallback_threshold)

match_df["_thin_content_threshold"] = match_df["Best_URL"].map(_thin_content_threshold_for_url)

def _thin_content_reason(row):
    if row["Recommendation"] not in ("No Action", "On-Page Optimization"):
        return ""
    wc = row["Matched_Page_Word_Count"]
    if pd.isna(wc):
        return ""
    reasons = []
    if wc < row["_thin_content_threshold"]:
        reasons.append(f"below site segment baseline ({row['_thin_content_threshold']:.0f} words)")
    if wc < COMPETITIVE_WORD_COUNT_THRESHOLD:
        reasons.append(f"below competitive-landscape threshold ({COMPETITIVE_WORD_COUNT_THRESHOLD} words)")
    if wc < THIN_CONTENT_ABSOLUTE_FLOOR:
        reasons.append(f"below absolute floor ({THIN_CONTENT_ABSOLUTE_FLOOR} words)")
    return "; ".join(reasons)

# Three independent checks, any one of which is enough to flag a page:
#   1. Below its own site-segment baseline (self-referential, Cell 6)
#   2. Below the editable COMPETITIVE_WORD_COUNT_THRESHOLD (a general
#      rule-of-thumb stand-in for real SERP data — see Cell 6 note)
#   3. Below THIN_CONTENT_ABSOLUTE_FLOOR (an unambiguous "too short
#      regardless of anything else" backstop)
# Thin_Content_Reason records exactly which check(s) fired, since with
# three separate conditions "why was this flagged" is worth being explicit
# about rather than leaving as a single opaque boolean.
match_df["Thin_Content_Reason"] = match_df.apply(_thin_content_reason, axis=1)
match_df["Thin_Content_Risk"] = match_df["Thin_Content_Reason"] != ""
match_df = match_df.drop(columns=["_thin_content_threshold"])

print(f"Expanded {len(cluster_match_df):,} cluster recommendations back out to "
      f"{len(match_df):,} individual keyword rows.")
print("\nRecommendation counts (per keyword):")
print(match_df["Recommendation"].value_counts().to_string())

thin_flag_count = match_df["Thin_Content_Risk"].sum()
print(f"\nThin-content risk: {thin_flag_count:,} keyword rows flagged "
      f"(competitive-landscape threshold: {COMPETITIVE_WORD_COUNT_THRESHOLD} words; "
      f"absolute floor: {THIN_CONTENT_ABSOLUTE_FLOOR} words; see per-segment baselines above).")

# Diagnostic: does the per-keyword INTRINSIC intent (Cell 9, classified from
# the keyword alone) agree with the per-CLUSTER CONTEXTUAL intent (Cell 13,
# classified from the keyword + its matched page)? A mismatch is worth a
# manual look — it can mean the matched page is subtly reframing the query,
# or that this keyword sits at the edge of its cluster.
match_df["Intent_Label_Agreement"] = (
    match_df["Keyword_Intent"] == match_df["Contextual_Keyword_Intent"]
)
disagreement_count = (~match_df["Intent_Label_Agreement"]).sum()
print(f"\nIntent label agreement: {len(match_df) - disagreement_count:,} agree, "
      f"{disagreement_count:,} disagree (intrinsic vs. contextual intent) — "
      f"worth spot-checking the disagreements.")

# --- Cluster-level summary: one row per cluster, for a content-planning view ---
cluster_keyword_lists = (
    cluster_df.groupby("Cluster_ID")["Keyword"]
    .apply(lambda s: ", ".join(sorted(s)))
    .rename("Cluster_Keywords")
    .reset_index()
)

# Aggregate search volume (sum) and difficulty (average) per cluster.
# min_count=1 on the sum means a cluster with NO volume data anywhere in it
# (either no column in the source file, or all values missing for this
# specific cluster) reports NaN rather than a misleading 0.
cluster_aggregates = cluster_df.groupby("Cluster_ID").agg(
    Cluster_Total_Search_Volume=("Search_Volume", lambda s: s.sum(min_count=1)),
    Cluster_Avg_Keyword_Difficulty=("Keyword_Difficulty", "mean"),
).reset_index()
cluster_aggregates["Cluster_Avg_Keyword_Difficulty"] = cluster_aggregates["Cluster_Avg_Keyword_Difficulty"].round(1)

_representative_content_lookup = (
    match_df[match_df["Keyword"] == match_df["Cluster_Representative_Keyword"]]
    [["Cluster_ID", "Matched_Page_Word_Count", "Thin_Content_Risk", "Thin_Content_Reason"]]
    .drop_duplicates(subset=["Cluster_ID"])
)

cluster_summary_df = cluster_match_df.merge(cluster_keyword_lists, on="Cluster_ID", how="left")
cluster_summary_df = cluster_summary_df.merge(cluster_aggregates, on="Cluster_ID", how="left")
cluster_summary_df = cluster_summary_df.merge(_representative_content_lookup, on="Cluster_ID", how="left")

summary_cols = [
    "Cluster_ID", "Cluster_Representative_Keyword", "Cluster_Keywords",
    "Cluster_Total_Search_Volume", "Cluster_Avg_Keyword_Difficulty",
    "Recommendation", "Recommendation_Reason",
    "Best_URL", "Semantic_Score", "Semantic_Bucket", "Literal_Score", "Literal_Bucket",
    "Contextual_Keyword_Intent", "Intent_Alignment", "Specificity_Alignment", "Intent_Reasoning",
    "Searcher_Intent_Summary",
    "Ambiguous_Match", "Cannibalization_Risk", "Shared_Best_URL_Cluster_Count",
    "Matched_Page_Word_Count", "Thin_Content_Risk", "Thin_Content_Reason",
]
cluster_summary_df = cluster_summary_df[summary_cols]
cluster_sizes_lookup = cluster_df.groupby("Cluster_ID")["Keyword"].count().rename("Cluster_Size")
cluster_summary_df = cluster_summary_df.merge(cluster_sizes_lookup, on="Cluster_ID", how="left")

# Highest-opportunity clusters first, when volume data is available — makes
# the exported Cluster Summary sheet a ready-made prioritization view.
if HAS_VOLUME_DATA:
    cluster_summary_df = cluster_summary_df.sort_values(
        "Cluster_Total_Search_Volume", ascending=False, na_position="last"
    ).reset_index(drop=True)

print(f"\nBuilt cluster-level summary: {len(cluster_summary_df):,} clusters.")
print(cluster_summary_df[["Cluster_Representative_Keyword", "Cluster_Size", "Recommendation"]].head(5).to_string(index=False))


In [ ]:
# ============================================================
# CELL 16 — EXPORT TO EXCEL
# Tab 1: Keyword Match Analysis — one row per keyword
# Tab 2: Cluster Summary — one row per thematic cluster
# ============================================================
OUTPUT_FILENAME = "keywordContentMatchAnalysis.xlsx"

RECOMMENDATION_COLORS = {
    "No Action": "#C6EFCE",
    "On-Page Optimization": "#FFEB9C",
    "New Content": "#FFC7CE",
}

def score_bg(score, min_s, max_s):
    if score is None or pd.isna(score):
        return "#FFFFFF"
    ratio = max(0, min(1, (score - min_s) / (max_s - min_s))) if max_s > min_s else 0
    if ratio < 0.5:
        r, g, b = 255, int(255 + (235 - 255) * (ratio / 0.5)), int(255 + (59 - 255) * (ratio / 0.5))
    else:
        r = int(255 + (76 - 255) * ((ratio - 0.5) / 0.5))
        g = int(235 + (175 - 235) * ((ratio - 0.5) / 0.5))
        b = int(59 + (80 - 59) * ((ratio - 0.5) / 0.5))
    return f"#{r:02X}{g:02X}{b:02X}"

def col_width(col):
    widths = {
        "Keyword": 30, "Best_URL": 55, "Runner_Up_URL": 55,
        "Cluster_Representative_Keyword": 30, "Cluster_Keywords": 60,
        "Recommendation": 20, "Recommendation_Reason": 40, "Intent_Reasoning": 45,
        "Searcher_Intent_Summary": 55,
        "Thin_Content_Reason": 45,
    }
    if col in widths:
        return widths[col]
    if "Score" in col or col.startswith("Literal_In"):
        return 14
    return 18

def write_sheet(workbook, writer, sheet_name, df, score_cols_100=(), score_cols_raw=()):
    ws = workbook.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws

    header_fmt = workbook.add_format({"bold": True, "bg_color": "#1F3864", "font_color": "#FFFFFF", "border": 1})
    for c, name in enumerate(df.columns):
        ws.write(0, c, name, header_fmt)

    score_ranges = {}
    for col in list(score_cols_100) + list(score_cols_raw):
        vals = df[col].dropna()
        score_ranges[col] = (float(vals.min()), float(vals.max())) if len(vals) else (0, 1)

    for r, (_, row) in enumerate(df.iterrows(), start=1):
        row_bg = "#FFFFFF" if r % 2 == 0 else "#F7F9FC"
        for c, col in enumerate(df.columns):
            val = row[col]
            if isinstance(val, float) and pd.isna(val):
                val = None
            if col == "Recommendation" and val in RECOMMENDATION_COLORS:
                fmt = workbook.add_format({"bg_color": RECOMMENDATION_COLORS[val], "bold": True})
                ws.write(r, c, val, fmt)
            elif col in score_ranges and val is not None:
                num_fmt = "0.0000" if col in score_cols_100 else "0.0"  # Literal_Score can be fractional with partial-match credit
                fmt = workbook.add_format({"num_format": num_fmt, "bg_color": score_bg(val, *score_ranges[col])})
                ws.write(r, c, val, fmt)
            elif isinstance(val, bool):
                fmt = workbook.add_format({"bg_color": row_bg})
                ws.write(r, c, "Yes" if val else "No", fmt)
            else:
                fmt = workbook.add_format({"bg_color": row_bg})
                if val is None:
                    ws.write_blank(r, c, None, fmt)
                else:
                    ws.write(r, c, val, fmt)

    for c, col in enumerate(df.columns):
        ws.set_column(c, c, col_width(col))

    ws.freeze_panes(1, 1)
    ws.autofilter(0, 0, len(df), len(df.columns) - 1)


keyword_export_cols = [
    "Keyword", "Recommendation", "Recommendation_Reason",
    "Cluster_ID", "Cluster_Representative_Keyword", "Cluster_Size",
    "Search_Volume", "Keyword_Difficulty",   # per-KEYWORD, from Cell 5 (if present in the source file)
    "Keyword_Intent",                 # intrinsic, per-KEYWORD (Cell 9) — not broadcast
    "Best_URL", "Semantic_Score", "Semantic_Bucket",
    "Literal_Score", "Literal_Bucket",
    "Literal_In_Title", "Literal_In_H1", "Literal_In_URL",
    "Literal_In_First100Words", "Literal_In_H2", "Literal_In_Body",
    "Contextual_Keyword_Intent",       # from Cell 13 — the page-context re-derived intent
    "Intent_Alignment", "Specificity_Alignment", "Intent_Reasoning",
    "Searcher_Intent_Summary",
    "Intent_Label_Agreement",          # intrinsic vs. contextual intent agree?
    "Ambiguous_Match", "Runner_Up_URL", "Runner_Up_Score",
    "Cannibalization_Risk", "Shared_Best_URL_Cluster_Count",
    "Matched_Page_Word_Count", "Thin_Content_Risk", "Thin_Content_Reason",
]
keyword_export_df = match_df[keyword_export_cols].copy()

with pd.ExcelWriter(OUTPUT_FILENAME, engine="xlsxwriter") as writer:
    workbook = writer.book

    write_sheet(
        workbook, writer, "Keyword Match Analysis", keyword_export_df,
        score_cols_100=["Semantic_Score", "Runner_Up_Score"],
        score_cols_raw=["Literal_Score"],
    )
    write_sheet(
        workbook, writer, "Cluster Summary", cluster_summary_df,
        score_cols_100=["Semantic_Score"],
        score_cols_raw=["Literal_Score"],
    )

print(f"Exported {len(keyword_export_df):,} keyword rows and {len(cluster_summary_df):,} cluster rows to '{OUTPUT_FILENAME}'.")
files.download(OUTPUT_FILENAME)


In [ ]:
# ============================================================
# CELL 17 (OPTIONAL) — SPOT CHECK A SINGLE KEYWORD (+ its cluster peers)
# ============================================================
test_keyword = "your test keyword here"  # ← change this

if test_keyword not in match_df["Keyword"].values:
    print("Keyword not found in results. Sample keywords:")
    print(match_df["Keyword"].head(5).tolist())
else:
    r = match_df[match_df["Keyword"] == test_keyword].iloc[0]
    print(f"Keyword: {r['Keyword']}")
    print(f"Cluster: {r['Cluster_ID']}  (representative: '{r['Cluster_Representative_Keyword']}', {r['Cluster_Size']} keywords)")
    print(f"Intrinsic keyword intent: {r['Keyword_Intent']}")
    print(f"Recommendation: {r['Recommendation']}  ({r['Recommendation_Reason']})")
    print(f"Best URL: {r['Best_URL']}")
    print(f"  Semantic: {r['Semantic_Score']} ({r['Semantic_Bucket']})")
    print(f"  Literal:  {r['Literal_Score']} ({r['Literal_Bucket']})")
    print(f"  Contextual intent (page-aware): {r['Contextual_Keyword_Intent']}")
    print(f"  Intent alignment: {r['Intent_Alignment']} / specificity {r['Specificity_Alignment']}")
    if not r["Intent_Label_Agreement"]:
        print(f"  \u26a0 Intrinsic intent ('{r['Keyword_Intent']}') disagrees with contextual intent ('{r['Contextual_Keyword_Intent']}') — worth a look")
    print(f"  Matched page word count: {r['Matched_Page_Word_Count']}")
    if r["Thin_Content_Risk"]:
        print(f"  \u26a0 Thin-content risk — {r['Thin_Content_Reason']}")
    print(f"  Reasoning: {r['Intent_Reasoning']}")
    print(f"  Searcher wants: {r['Searcher_Intent_Summary']}")
    if r["Cannibalization_Risk"]:
        print(f"  ⚠ Cannibalization risk — {r['Shared_Best_URL_Cluster_Count']} clusters share this best-match URL")

    peers = match_df[(match_df["Cluster_ID"] == r["Cluster_ID"]) & (match_df["Keyword"] != test_keyword)]["Keyword"].tolist()
    if peers:
        print(f"\nOther keywords in this cluster ({len(peers)}):")
        for p in peers[:15]:
            print(f"    - {p}")
        if len(peers) > 15:
            print(f"    ... and {len(peers) - 15} more")
